In [51]:
import torch
import torch.nn.functional as F
import os 
from torchvision import transforms
from PIL import Image


In [52]:
# Load DINO ViT-S/16 pretrained
model = torch.hub.load('facebookresearch/dino:main', 'dino_vits16')
model.eval()

Using cache found in /home/saad/.cache/torch/hub/facebookresearch_dino_main


VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (blocks): ModuleList(
    (0-11): 12 x Block(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (norm): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
  (head): Identity()
)

In [53]:


transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


img = Image.open("../data/exp_data/cat_01.jpeg").convert("RGB")
x = transform(img).unsqueeze(0)  # shape: [1, 3, 224, 224]


In [66]:

# PRE-PROCESSING MULTIPLE IMAGES AND EXTRACTING CLS EMBEDDINGS
path = "../data/exp_data/"
cls_embeddings = []
for image in os.listdir(path):
    try:
        img = Image.open(os.path.join(path, image)).convert("RGB")
        x = transform(img).unsqueeze(0)  # shape: [1, 3, 224, 224]
        with torch.no_grad():
            cls_embedding = model.get_intermediate_layers(x, n=2)[0][:, 0]
            cls_embedding = F.normalize(cls_embedding, dim=-1)
            cls_embeddings.append((image, cls_embedding))

    except:
        print(f"Could not process image: {image}")
        continue

Could not process image: dog_02.avif


In [67]:
emb1 = cls_embeddings[0][1]  # cat
emb2 = cls_embeddings[1][1]  # dog 
emb3 = cls_embeddings[2][1]  # cat
emb4 = cls_embeddings[3][1]  # cat
emb5 = cls_embeddings[4][1]  # dog

print(torch.cosine_similarity(emb1, emb2))  # should be low
print(torch.cosine_similarity(emb1, emb3))  # should be high
print(torch.cosine_similarity(emb1, emb4))  # should be high
print(torch.cosine_similarity(emb1, emb5))  # should be low

tensor([0.2815])
tensor([0.7061])
tensor([0.6694])
tensor([0.2229])


In [68]:
len(emb1[0])

384